# Notebook 3 - Classification Challenge (Bank Marketing)

**Expected duration:** 60-90 minutes

## Objective
Independently build and evaluate binary classifiers for term-deposit subscription prediction.

## Inputs
- UCI Bank Marketing data (`data/bank-full.csv`, auto-downloaded if missing)
- Mixed numeric and categorical customer/contact features

## Outputs
- Checked challenge milestones with grading progress
- Model comparison table with F1 and ROC-AUC

## Checkpoint expectations
- Pass 9 guided checks (`todo_1` to `todo_9`)
- Compare baseline, logistic, random forest, tuned forest
- Prepare submission probabilities (`id,y_prob`) for leaderboard


## Dataset handling
The notebook downloads the UCI Bank Marketing dataset on first run and caches `bank-full.csv` at `data/bank-full.csv`.

For an offline workshop, download `bank-full.csv` in advance and place it there.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn ipython


## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import json
import io, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, KFold, StratifiedKFold
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
def load_solution_bank(path=None):
    candidates = []
    if path is not None:
        candidates.append(Path(path))

    cwd = Path.cwd()
    rel = Path('.instructor') / 'bank_challenge_solutions.json'
    candidates.extend([
        cwd / rel,
        cwd.parent / rel,
        cwd.parent.parent / rel,
        Path(rel),
    ])

    seen = set()
    for cand in candidates:
        try:
            resolved = cand.resolve(strict=False)
        except Exception:
            resolved = cand
        key = str(resolved)
        if key in seen:
            continue
        seen.add(key)

        try:
            if cand.exists():
                with cand.open('r', encoding='utf-8') as f:
                    data = json.load(f)
                if isinstance(data, dict):
                    return data
        except Exception as exc:
            print(f'Warning: could not load solution bank from {cand}: {exc}')

    return {}


SOLUTION_BANK = load_solution_bank()


def reveal_solution(solution_key):
    answer = SOLUTION_BANK.get(solution_key)
    if answer is None:
        print('Solution is not available in this copy. Ask the instructor after attempting the exercise.')
        return
    display(Markdown('**Reference answer:**'))
    display(Markdown('```python\n' + answer.strip() + '\n```'))


def _check_todo_1(ns):
    return (
        ns.get('bank_shape') == ns.get('bank').shape
        and ns.get('bank_columns') == list(ns.get('bank').columns)
        and isinstance(ns.get('target_distribution'), pd.Series)
    )


def _check_todo_2(ns):
    X = ns.get('X')
    y = ns.get('y')
    return (
        isinstance(X, pd.DataFrame)
        and 'y' not in X.columns
        and isinstance(y, pd.Series)
        and set(y.dropna().unique()) <= {0, 1}
        and y.name == 'y'
    )


def _check_todo_3(ns):
    X = ns.get('X')
    if not isinstance(X, pd.DataFrame):
        return False
    expected_num = X.select_dtypes(include=[np.number]).columns.tolist()
    expected_cat = X.select_dtypes(exclude=[np.number]).columns.tolist()
    return ns.get('numeric_features') == expected_num and ns.get('categorical_features') == expected_cat


def _check_todo_4(ns):
    X_train = ns.get('X_train')
    X_test = ns.get('X_test')
    y_train = ns.get('y_train')
    y_test = ns.get('y_test')
    if any(v is None for v in [X_train, X_test, y_train, y_test]):
        return False
    return (
        X_train.shape[0] == 36168
        and X_test.shape[0] == 9043
        and abs(float(y_train.mean()) - float(y_test.mean())) < 0.01
    )


def _check_todo_5(ns):
    numeric_pipeline = ns.get('numeric_pipeline')
    categorical_pipeline = ns.get('categorical_pipeline')
    if numeric_pipeline is None or categorical_pipeline is None:
        return False
    return (
        numeric_pipeline.named_steps['imputer'].strategy == 'median'
        and categorical_pipeline.named_steps['imputer'].strategy == 'most_frequent'
        and categorical_pipeline.named_steps['encoder'].handle_unknown == 'ignore'
    )


def _check_todo_6(ns):
    baseline_clf = ns.get('baseline_clf')
    if baseline_clf is None:
        return False
    model = baseline_clf.named_steps.get('model')
    return isinstance(model, DummyClassifier) and model.strategy == 'most_frequent'


def _check_todo_7(ns):
    logistic_model = ns.get('logistic_model')
    if logistic_model is None:
        return False
    model = logistic_model.named_steps.get('model')
    return (
        isinstance(model, LogisticRegression)
        and model.max_iter == 2000
        and model.class_weight == 'balanced'
    )


def _check_todo_8(ns):
    forest_clf = ns.get('forest_clf')
    if forest_clf is None:
        return False
    est = forest_clf.named_steps.get('model')
    return isinstance(est, RandomForestClassifier) and est.n_estimators == 150 and est.class_weight == 'balanced'


def _check_todo_9(ns):
    clf_search = ns.get('clf_search')
    return (
        isinstance(clf_search, RandomizedSearchCV)
        and clf_search.n_iter == 8
        and clf_search.scoring == 'f1'
        and hasattr(clf_search, 'best_estimator_')
    )


CHECKS = {
    'todo_1': ('dataset inspection is correct.', _check_todo_1),
    'todo_2': ('features and binary target are correct.', _check_todo_2),
    'todo_3': ('feature groups are correct.', _check_todo_3),
    'todo_4': ('the stratified split is correct.', _check_todo_4),
    'todo_5': ('preprocessing is correct.', _check_todo_5),
    'todo_6': ('baseline classifier is correct.', _check_todo_6),
    'todo_7': ('logistic regression classifier is correct.', _check_todo_7),
    'todo_8': ('random forest classifier is correct.', _check_todo_8),
    'todo_9': ('classification tuning is configured correctly.', _check_todo_9),
}

CHECK_ORDER = list(CHECKS.keys())
CHECK_POINTS = {key: 1 for key in CHECK_ORDER}
GRADEBOOK = {key: False for key in CHECK_ORDER}


def _score_snapshot():
    earned = sum(CHECK_POINTS[k] for k, done in GRADEBOOK.items() if done)
    total = sum(CHECK_POINTS.values())
    return earned, total


def mark_summary(show_table=True):
    earned, total = _score_snapshot()
    pct = (100.0 * earned / total) if total else 0.0
    print(f'Challenge checks passed: {earned}/{total} ({pct:.1f}%)')

    if show_table:
        rows = []
        for key in CHECK_ORDER:
            rows.append({
                'task': key,
                'status': 'passed' if GRADEBOOK[key] else 'pending',
                'points': CHECK_POINTS[key],
            })
        display(pd.DataFrame(rows))


def check_and_reveal(solution_key):
    if solution_key not in CHECKS:
        print(f'Unknown solution key: {solution_key}')
        return

    message, checker = CHECKS[solution_key]
    try:
        is_correct = bool(checker(globals()))
    except Exception:
        is_correct = False

    if not is_correct:
        print('Not correct yet. Revise the previous cell and rerun this check.')
        return

    GRADEBOOK[solution_key] = True
    print('Correct - ' + message)
    reveal_solution(solution_key)

    earned, total = _score_snapshot()
    print(f'Marking progress: {earned}/{total} checks passed')


def close_enough(a, b, tol=1e-8):
    try:
        return abs(float(a) - float(b)) <= tol
    except Exception:
        return False


def _extract_csv_bytes_from_zip(raw_zip, suffix):
    """Return (csv_bytes, member_name) from a zip payload, searching nested zips if needed."""
    suffix_lower = suffix.lower()

    with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
        names = zf.namelist()

        direct = [name for name in names if name.lower().endswith(suffix_lower)]
        if direct:
            selected = direct[0]
            return zf.read(selected), selected

        nested_zips = [name for name in names if name.lower().endswith('.zip')]
        for nested_name in nested_zips:
            try:
                nested_bytes = zf.read(nested_name)
                found = _extract_csv_bytes_from_zip(nested_bytes, suffix)
                if found is not None:
                    return found
            except zipfile.BadZipFile:
                continue

        csv_candidates = [name for name in names if name.lower().endswith('.csv')]
        if csv_candidates:
            def _score(name):
                leaf = Path(name).name.lower()
                info = zf.getinfo(name)
                return (
                    int('bank-full' in leaf),
                    int('full' in leaf),
                    info.file_size,
                )

            selected = max(csv_candidates, key=_score)
            return zf.read(selected), selected

    return None


def load_csv_from_uci_zip(url, suffix, local_path, sep=','):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)

    if local_path.exists():
        return pd.read_csv(local_path, sep=sep)

    try:
        with urllib.request.urlopen(url) as resp:
            raw_zip = resp.read()

        found = _extract_csv_bytes_from_zip(raw_zip, suffix)
        if found is None:
            raise FileNotFoundError(f'{suffix} not found in archive (including nested archives).')

        csv_bytes, member_name = found
        local_path.write_bytes(csv_bytes)
        print(f'Loaded from archive member: {member_name}')
        print(f'Cached dataset to: {local_path}')
        return pd.read_csv(local_path, sep=sep)

    except Exception as exc:
        raise RuntimeError(
            f'Automatic dataset download failed. Place the file at {local_path} and rerun. Original error: {exc}'
        ) from exc


### Solution reveal and marking policy
After you run each check cell, the solution is automatically shown **only if your answer is correct**.

Each passed check is recorded in a gradebook. Use `mark_summary()` any time to see progress.


## 1. Load the dataset

In [ ]:
BANK_URL = 'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip'
bank = load_csv_from_uci_zip(BANK_URL, 'bank-full.csv', 'data/bank-full.csv', sep=';')
bank.head()

## 2. Inspect the data
### Exercise 1
Create `bank_shape`, `bank_columns`, and `target_distribution`.

In [ ]:
# TODO 1
bank_shape = bank.________
bank_columns = list(bank.________)
target_distribution = bank['y'].________(normalize=True)

bank_shape, bank_columns[:5], target_distribution

In [ ]:
check_and_reveal('todo_1')


## 3. Define target and features
Target: `y` where `yes` means subscribed and `no` means not subscribed.

In [ ]:
# TODO 2
X = bank.________(columns='y')
y = bank['y'].________({'no':0, 'yes':1})

X.shape, y.value_counts()

In [ ]:
check_and_reveal('todo_2')


## 4. Identify feature types

In [ ]:
# TODO 3
numeric_features = X.________(include=[np.number]).columns.tolist()
categorical_features = X.________(exclude=[np.number]).columns.tolist()

numeric_features, categorical_features

In [ ]:
check_and_reveal('todo_3')


## 5. Split data with stratification

In [ ]:
# TODO 4
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=____, random_state=________, stratify=________
)

y_train.mean(), y_test.mean(), X_train.shape, X_test.shape

In [ ]:
check_and_reveal('todo_4')


## 6. Build preprocessing

In [ ]:
# TODO 5
numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='________')),
    ('scaler', StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='________')),
    ('encoder', OneHotEncoder(handle_unknown='________'))
])
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])
preprocessor

In [ ]:
check_and_reveal('todo_5')


## 7. Baseline classifier
Use a most-frequent baseline so accuracy can be interpreted against a trivial benchmark.

In [ ]:
# TODO 6
baseline_clf = Pipeline(steps=[
    ('preprocessor', ________),
    ('model', DummyClassifier(strategy='________'))
])
baseline_clf.fit(X_train, y_train)
baseline_pred = baseline_clf.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_accuracy

In [ ]:
check_and_reveal('todo_6')


## 8. Logistic regression classifier

In [ ]:
# TODO 7
logistic_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=____, class_weight=____))
])
logistic_model.fit(X_train, y_train)
logistic_pred = logistic_model.predict(X_test)
logistic_prob = logistic_model.predict_proba(X_test)[:, 1]

logistic_accuracy = accuracy_score(y_test, logistic_pred)
logistic_precision = precision_score(y_test, logistic_pred)
logistic_recall = recall_score(y_test, logistic_pred)
logistic_f1 = f1_score(y_test, logistic_pred)
logistic_auc = roc_auc_score(y_test, logistic_prob)

logistic_accuracy, logistic_precision, logistic_recall, logistic_f1, logistic_auc

In [ ]:
check_and_reveal('todo_7')


## 9. Confusion matrix and metric interpretation

In [ ]:
cm = confusion_matrix(y_test, logistic_pred)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No subscription','Subscription']).plot()
plt.title('Logistic Regression Confusion Matrix')
plt.show()

print(classification_report(y_test, logistic_pred, target_names=['No','Yes']))

### Reflection prompt
For this marketing problem, which metric would you prioritize: precision, recall, F1, or ROC-AUC? Defend your choice.

## 10. Random forest classifier

In [ ]:
# TODO 8
forest_clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=____,
        max_depth=____,
        class_weight=____,
        random_state=________,
        n_jobs=____
    ))
])
forest_clf.fit(X_train, y_train)
forest_pred = forest_clf.predict(X_test)
forest_prob = forest_clf.predict_proba(X_test)[:, 1]

forest_f1 = f1_score(y_test, forest_pred)
forest_auc = roc_auc_score(y_test, forest_prob)
forest_f1, forest_auc

In [ ]:
check_and_reveal('todo_8')


## 11. Hyperparameter tuning challenge
Tune the random forest using stratified cross-validation. Optimise for `f1` rather than accuracy.

In [ ]:
# TODO 9
param_distributions = {
    'model__n_estimators':[80,120,180],
    'model__max_depth':[None,8,14,20],
    'model__min_samples_split':[2,5,10],
    'model__min_samples_leaf':[1,2,4],
    'model__max_features':['sqrt',0.7,1.0],
    'model__class_weight':['balanced','balanced_subsample'],
}

tuning_clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))
])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
clf_search = RandomizedSearchCV(
    tuning_clf, param_distributions=param_distributions, n_iter=____,
    scoring='____', cv=cv, random_state=________, n_jobs=-1, verbose=1
)
clf_search.fit(X_train, y_train)

In [ ]:
check_and_reveal('todo_9')


## 12. Evaluate the tuned classifier

In [ ]:
best_clf = clf_search.best_estimator_
tuned_pred = best_clf.predict(X_test)
tuned_prob = best_clf.predict_proba(X_test)[:,1]

metrics = pd.DataFrame({
    'model':['Baseline','Logistic regression','Random forest','Tuned random forest'],
    'accuracy':[baseline_accuracy, logistic_accuracy, accuracy_score(y_test, forest_pred), accuracy_score(y_test, tuned_pred)],
    'precision':[np.nan, logistic_precision, precision_score(y_test, forest_pred), precision_score(y_test, tuned_pred)],
    'recall':[np.nan, logistic_recall, recall_score(y_test, forest_pred), recall_score(y_test, tuned_pred)],
    'f1':[np.nan, logistic_f1, forest_f1, f1_score(y_test, tuned_pred)],
    'roc_auc':[np.nan, logistic_auc, forest_auc, roc_auc_score(y_test, tuned_prob)]
})
metrics


## 13. Marking summary
Run this section to view grading progress and current model metric ranking.


In [ ]:
mark_summary()

if 'metrics' in globals() and isinstance(metrics, pd.DataFrame):
    print("\nModel metric view (for competition ranking reference):")
    display(metrics[['model','f1','roc_auc']].sort_values('f1', ascending=False))


## 14. Final challenge questions
